# Notebook 5: Community Detection and Aggregation

This notebook processes the attribute-enriched graph **G2** to detect structural communities using the **Leiden algorithm**.

For each community, the LLM is invoked to generate a **community summary** using all semantic units within that community.  
If a community contains only one semantic unit, summarization is skipped.  
These summaries form the content for **High-level Element (H) nodes**.

From each community summary, an additional LLM call extracts **keywords** and **titles**.  
These results serve as the content for **High-level Overview (O) nodes**.

The resulting communities are also saved for future processing.

In [1]:
import os
dir_path = os.getcwd()
print("The directory of this script is:", dir_path)
root_path = os.path.dirname(dir_path)
print("The root directory is:", root_path)

The directory of this script is: d:\Documents\GitHub\NodeRAG\graphs
The root directory is: d:\Documents\GitHub\NodeRAG


In [2]:
import sys
sys.path.append(root_path)
from graphs.Node import Node

In [3]:
import pickle
with open(f"{root_path}/graphs/data/graphs/G2_medical_attribute_enriched_graph.pkl", "rb") as f:
    medical_g2 = pickle.load(f)

In [4]:
import igraph as ig
from leidenalg import find_partition, ModularityVertexPartition
from collections import defaultdict

def leiden_community_detection(node_dict):
    id_to_idx = {nid: i for i, nid in enumerate(node_dict.keys())}
    idx_to_id = {v: k for k, v in id_to_idx.items()}

    edge_weights = defaultdict(float)
    for nid, node in node_dict.items():
        for neighbor, weight in node.edges.items():
            a, b = id_to_idx[nid], id_to_idx[neighbor]
            if a < b:
                edge_weights[(a, b)] += weight

    edges = list(edge_weights.keys())
    weights = list(edge_weights.values())

    g = ig.Graph(n=len(node_dict), edges=edges, directed=False)
    g.es["weight"] = weights

    partition = find_partition(g, ModularityVertexPartition, weights="weight")
    membership = partition.membership
    communities = {idx_to_id[i]: membership[i] for i in range(len(node_dict))}
    return communities


In [5]:
from collections import Counter
medical_communities = leiden_community_detection(medical_g2)
medical_communities_counts = Counter(medical_communities.values())
s1 = 0
for val in medical_communities_counts.values():
    if val==1:
        s1+=val
print("Medical single-node communities:", s1)

Medical single-node communities: 187


In [6]:
medical_communities_counts

Counter({0: 2428,
         1: 2195,
         2: 2177,
         3: 2130,
         4: 1894,
         5: 1886,
         6: 1876,
         7: 1475,
         8: 1403,
         9: 1381,
         10: 1371,
         11: 1363,
         12: 1362,
         13: 1322,
         14: 1299,
         15: 1186,
         16: 1178,
         17: 1082,
         18: 1077,
         19: 1068,
         20: 1064,
         21: 1054,
         22: 996,
         23: 982,
         24: 936,
         25: 918,
         26: 862,
         27: 817,
         28: 677,
         29: 521,
         30: 407,
         31: 353,
         32: 349,
         33: 249,
         34: 228,
         35: 211,
         36: 190,
         37: 190,
         38: 155,
         39: 45,
         40: 29,
         41: 8,
         42: 5,
         43: 4,
         44: 4,
         45: 3,
         46: 1,
         47: 1,
         48: 1,
         49: 1,
         50: 1,
         51: 1,
         52: 1,
         53: 1,
         54: 1,
         55: 1,
         56:

In [7]:
communities_medical = defaultdict(list)
for node_id, community_id in medical_communities.items():
    #if medical_g2[node_id].node_type == "N":
    #    continue
    communities_medical[community_id].append(node_id)

In [8]:
communities_medical

defaultdict(list,
            {17: ['medical-0-S-0',
              'medical-0-S-0-R-0',
              'medical-0-S-0-R-1',
              'medical-0-S-0-R-2',
              'medical-0-S-0-R-3',
              'medical-0-S-0-R-4',
              'medical-0-S-1',
              'medical-0-S-1-R-0',
              'medical-0-S-1-R-1',
              'medical-0-S-1-R-2',
              'medical-0-S-1-R-3',
              'medical-0-S-1-R-4',
              'medical-0-S-1-R-5',
              'medical-0-S-1-R-6',
              'medical-0-S-1-R-7',
              'medical-0-S-1-R-8',
              'medical-0-S-1-R-9',
              'medical-0-S-1-R-10',
              'medical-0-S-1-R-11',
              'medical-0-S-1-R-12',
              'medical-0-S-2-R-1',
              'medical-0-S-2-R-2',
              'medical-0-S-3',
              'medical-0-S-3-R-0',
              'medical-0-S-3-R-1',
              'medical-0-S-3-R-2',
              'medical-0-S-3-R-3',
              'medical-0-S-3-R-4',
       

In [9]:
from google import genai

with open(f"{root_path}/API_KEY.txt", "r", encoding="utf-8") as f:
    API_KEY = f.read()
    
def call_gemini(text):
    client = genai.Client(api_key = API_KEY)
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=text,
    )
    return response.text

In [10]:
import time
from tqdm import tqdm

In [11]:
medical_communities_response = {}

In [12]:
from graphs.prompt.high_level_elements_generation_prompt import high_level_elements_generation_prompt

for c in tqdm(communities_medical):
    if f'medical-H-{c}' in medical_communities_response:
        continue
    semantic_count = 0
    nodes_id = communities_medical[c]
    content = []
    for nid in nodes_id:
        node = medical_g2[nid]
        if node.node_type == "S":
            semantic_count+=1
        if node.node_type in ["R", "N"]:
            continue
        content.append(node.content)
    if semantic_count < 2:
        continue
    else:
        content = "\n".join(content)
        response = call_gemini(high_level_elements_generation_prompt(content))
        medical_communities_response[f'medical-H-{c}'] = response
        time.sleep(2)

100%|██████████| 233/233 [17:04<00:00,  4.40s/it] 


In [13]:
medical_communities_overview = {}

In [14]:
from graphs.prompt.high_level_overview_generation_prompt import high_level_overview_prompt
for element_id, element in tqdm(medical_communities_response.items()):
    overview_id = f"{element_id}-O-0"
    if overview_id in medical_communities_overview:
        continue
    overview = call_gemini(high_level_overview_prompt(element))
    medical_communities_overview[overview_id] = overview

100%|██████████| 41/41 [04:27<00:00,  6.53s/it]


In [15]:
import pandas as pd
medical_df = pd.DataFrame(list(medical_communities_response.items()), columns=["node_id", "community_summary"])
medical_df.to_parquet("data//nodes/community/medical_communities_summary.parquet")
medical_df = pd.read_parquet("data/nodes/community/medical_communities_summary.parquet")
medical_df

,node_id,community_summary
0,medical-H-17,Here are distinct categories of high-level inf...
1,medical-H-11,Here are five distinct categories of high-leve...
2,medical-H-20,Here are the distinct categories of high-level...
3,medical-H-7,Here are distinct categories of high-level inf...
4,medical-H-27,Here are distinct categories of high-level inf...
5,medical-H-1,Here are distinct categories of high-level inf...
6,medical-H-23,Here are distinct categories of high-level inf...
7,medical-H-22,Here are distinct categories of high-level inf...
8,medical-H-4,Here are distinct categories of high-level inf...
9,medical-H-6,Here are distinct categories of high-level inf...


In [16]:
medical_overview = pd.DataFrame(list(medical_communities_overview.items()), columns=["node_id", "community_overview"])
medical_overview.to_parquet("data/nodes/community/medical_communities_overview.parquet")
medical_overview = pd.read_parquet("data/nodes/community/medical_communities_overview.parquet")
medical_overview

,node_id,community_overview
0,medical-H-17-O-0,"Skin Cancer: Basal Cell Carcinoma, Squamous Ce..."
1,medical-H-11-O-0,"Head and Neck Cancer: Anatomy, Diagnosis, Trea..."
2,medical-H-20-O-0,"Cancer Management: Surgical Treatment, Staging..."
3,medical-H-7-O-0,"Lymphatic System, Cancer Metastasis, Breast Ca..."
4,medical-H-27-O-0,"NCCN Guidelines: Cancer Patient Empowerment, P..."
5,medical-H-1-O-0,"Cancer Genetics, Mutations, Genomic Testing, a..."
6,medical-H-23-O-0,"Cancer: Risk Factors, Pathogenesis, Staging, S..."
7,medical-H-22-O-0,"HPV-Related Cancers, Biomarkers, Treatment Sid..."
8,medical-H-4-O-0,"Patient Health Management, Advocacy, Support"
9,medical-H-6-O-0,"Cancer Care: Diagnostics, Personalized Treatme..."


In [17]:
with open(f"{root_path}/graphs/data/nodes/community/medical_communities.pkl", "wb") as f:
    pickle.dump(communities_medical, f)
with open(f"{root_path}/graphs/data/nodes/community/medical_communities.pkl", "rb") as f:
    communities_medical_loaded = pickle.load(f)

In [18]:
communities_medical_loaded

defaultdict(list,
            {17: ['medical-0-S-0',
              'medical-0-S-0-R-0',
              'medical-0-S-0-R-1',
              'medical-0-S-0-R-2',
              'medical-0-S-0-R-3',
              'medical-0-S-0-R-4',
              'medical-0-S-1',
              'medical-0-S-1-R-0',
              'medical-0-S-1-R-1',
              'medical-0-S-1-R-2',
              'medical-0-S-1-R-3',
              'medical-0-S-1-R-4',
              'medical-0-S-1-R-5',
              'medical-0-S-1-R-6',
              'medical-0-S-1-R-7',
              'medical-0-S-1-R-8',
              'medical-0-S-1-R-9',
              'medical-0-S-1-R-10',
              'medical-0-S-1-R-11',
              'medical-0-S-1-R-12',
              'medical-0-S-2-R-1',
              'medical-0-S-2-R-2',
              'medical-0-S-3',
              'medical-0-S-3-R-0',
              'medical-0-S-3-R-1',
              'medical-0-S-3-R-2',
              'medical-0-S-3-R-3',
              'medical-0-S-3-R-4',
       